# RubricSpan 训练（Kaggle）

前置条件：
1. Settings → Accelerator 选 **GPU T4 x2**（或 P100）；Internet → **ON**（需手机验证）；
2. Attach 已上传的私有数据集 `rubricspan-data`（Add Input → 你的 Datasets）；
3. Secrets（Add-ons → Secrets）按需添加：
   - `LLM_ENDPOINT_1_BASE_URL` / `LLM_ENDPOINT_1_API_KEY` / `LLM_ENDPOINT_1_MODEL`（打标用，多个端点递增编号）
   - `HF_TOKEN`（发布模型回 Hugging Face 时用）

配额：GPU 约 30 小时/周；单次会话最长 12 小时——长任务用 **Save & Run All（后台）**，
产物落 `/kaggle/working` 随版本保存（上限约 20GB）。

In [ ]:
import os, shutil
from pathlib import Path

# 数据挂载点（按标记文件定位，不拘泥于目录名）
inp = Path("/kaggle/input")
markers = sorted(inp.rglob("mrc_train.jsonl"))
assert markers, f"mrc_train.jsonl not found under {inp}"
data_mount = markers[0].parent.parent
os.environ["RUBRICSPAN_DATA_DIR"] = str(data_mount)

# 模型产物目录（可写，随版本保存）
os.environ["RUBRICSPAN_MODELS_DIR"] = "/kaggle/working/models"
print("data:", os.environ["RUBRICSPAN_DATA_DIR"])
print("models:", os.environ["RUBRICSPAN_MODELS_DIR"])

In [ ]:
# 从 rubricspan-repo 数据集复制源码到 working（挂载区只读）
import os, shutil, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/RubricSpan")
if not REPO_DIR.exists():
    inp = Path("/kaggle/input")
    markers = sorted(inp.rglob("pyproject.toml"))
    assert markers, "pyproject.toml not found (attach rubricspan-repo?)"
    repo_src = markers[0].parent.parent
    shutil.copytree(str(repo_src), str(REPO_DIR))
    print(f"code copied from {repo_src}")

sys.path.insert(0, str(REPO_DIR / "train"))

# 安装未预装的依赖
reqs = (REPO_DIR / "train" / "requirements.txt").read_text().splitlines()
extras = []
for r in reqs:
    r = r.strip()
    if not r or r.startswith("#"):
        continue
    pkg = r.split(">=")[0].split("==")[0].split("[")[0].strip()
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        extras.append(r)
if extras:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + extras)
    print(f"installed {len(extras)} extra(s)")
else:
    print("all deps already satisfied")

os.chdir(str(REPO_DIR / "train"))
print("setup done")

In [ ]:
# 基座权重：从 HuggingFace 拉到本地 backbone 布局
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from sentence_transformers import SentenceTransformer
from pathlib import Path
import os

models_dir = Path(os.environ["RUBRICSPAN_MODELS_DIR"])
backbone = models_dir / "backbone"
backbone.mkdir(parents=True, exist_ok=True)

# text2vec-base-chinese
dst = backbone / "text2vec-base-chinese"
if not dst.exists():
    print("downloading text2vec-base-chinese ...")
    m = SentenceTransformer("shibing624/text2vec-base-chinese", device="cpu")
    m.save_pretrained(str(dst))
    print("text2vec saved")

# mengzi-bert-base
dst = backbone / "mengzi-bert-base"
if not dst.exists():
    print("downloading mengzi-bert-base ...")
    tok = AutoTokenizer.from_pretrained("langboat/mengzi-bert-base")
    model = AutoModelForQuestionAnswering.from_pretrained("langboat/mengzi-bert-base")
    tok.save_pretrained(str(dst))
    model.save_pretrained(str(dst))
    print("mengzi saved")
print("backbones ready")

## 训练 / 评估 / 导出

按需执行下面各格。全部命令在 `train/` 目录语义下运行（与仓库指南一致）。

In [ ]:
import os, torch


# M2 训练：MRC（bridge+main；T4/P100 自动 fp16，Ampere+ 为 bf16）
%cd {REPO_DIR}/train
!CUDA_VISIBLE_DEVICES="" python -m rubricspan_train.training.train_mrc --stage both --epochs 4

In [ ]:
# M2 训练：相似度
%cd {REPO_DIR}/train
!python -m rubricspan_train.training.train_similarity

In [ ]:
# 评估报告
%cd {REPO_DIR}/train
!python -m rubricspan_train.evaluation.report

In [ ]:
# ONNX 导出 + 量化 + 校验（CPU 即可）
%cd {REPO_DIR}/train
!python -m rubricspan_train.export.onnx export
!python -m rubricspan_train.export.onnx quantize
!python -m rubricspan_train.export.onnx verify

## （可选）发布回 Hugging Face

先在本地跑 `scripts/prepare_model_repos.py` 的产物布局对齐流程，或直接把
`/kaggle/working/models/{mrc,similarity}` 下的 onnx + tokenizer 上传到既有厂库。

In [ ]:
import os
from pathlib import Path
from huggingface_hub import upload_folder

HF_USER = "Maicarons"  # 改成你的 HF 用户名
token = os.environ.get("HF_TOKEN")  # UI 运行来自 Kaggle Secret；API 推送的运行没有则跳过
models = Path(os.environ["RUBRICSPAN_MODELS_DIR"])
for sub, repo in {"mrc": "rubricspan-mrc-onnx", "similarity": "rubricspan-similarity-onnx"}.items():
    src = models / sub
    if not src.exists():
        continue
    if not token:
        print(f"skip {repo}: no HF_TOKEN secret")
        continue
    upload_folder(folder_path=str(src), repo_id=f"{HF_USER}/{repo}",
                  commit_message="train on kaggle", token=token)